# Tutorial: Polars Exercise Solutions

Audience:
- Instructors or learners reviewing worked solutions for the short exercises in [`EXERCISES.md`](EXERCISES.md).

Prerequisites:
- Basic Python syntax.
- A working Polars environment with Excel support.

Learning goals:
- Load tabular data from Excel and CSV with Polars.
- Filter, summarize, reshape, and clean data with Polars expressions.
- Compare Polars workflows to the parallel pandas exercises.


## Outline

1. Setup and shared data
2. Basic Polars objects and selection
3. Missing data and cleanup
4. Aggregation, pivoting, and reshaping
5. Expressions, chaining, and mini analysis


In [ ]:
from pathlib import Path
from io import StringIO

import matplotlib.pyplot as plt
import polars as pl

DATA_DIR = Path('data')
metadata = pl.read_excel(DATA_DIR / 'patient_metadata.xlsx')
experiment = pl.read_excel(DATA_DIR / 'patient_experiment.xlsx')
patients = experiment.join(metadata, on='patient', how='left')

print('metadata', metadata.shape)
print('experiment', experiment.shape)
patients.head()


## 1. Create a Series

Create a simple Polars series and inspect summary statistics.


In [ ]:
s = pl.Series('values', [3, 5, 8, 13, 21])
print(s)
print('mean', s.mean())
print('min', s.min())
print('max', s.max())


## 2. Build a DataFrame

Build a small frame and derive one new column with `with_columns()`.


In [ ]:
df = pl.DataFrame(
    {
        'name': ['Alice', 'Bob', 'Carol'],
        'age': [28, 34, 41],
        'city': ['Brussels', 'Antwerp', 'Ghent'],
    }
).with_columns((pl.col('age') + 10).alias('age_in_10_years'))

df


## 3. Inspect Patient Metadata

Read the patient metadata and inspect shape, schema, and head.


In [ ]:
print(metadata.shape)
print(metadata.schema)
metadata.head()


## 4. Filter Rows with Expressions

Join the experiment and metadata tables, then filter the combined table.


In [ ]:
high_temp = patients.filter(pl.col('temperature') > 38.5)
condition_a = patients.filter(pl.col('condition') == 'A')
high_temp_f = patients.filter(
    (pl.col('temperature') > 38.5) & (pl.col('gender') == 'F')
)

print('high temperature rows:', high_temp.height)
print('condition A rows:', condition_a.height)
high_temp_f.head()


## 5. Select Rows and Columns

Use Polars selection helpers instead of pandas-style indexing.


In [ ]:
first_rows = metadata.select(['patient', 'gender']).head(4)
gender_series = metadata.get_column('gender')

print(first_rows)
print(gender_series)


## 6. Missing-Value Count

Create a small frame with nulls and count them per column.


In [ ]:
missing = pl.DataFrame(
    {
        'int_data': [3, 5, None, 17],
        'float_data': [3.7, None, 7.5, 3.5],
        'category_data': ['A', 'A', None, 'B'],
        'string_data': ['str1', None, 'str2', 'str3'],
    }
)
missing.null_count()


## 7. Fill or Drop Missing Values

Show two cleanup strategies on the same small frame.


In [ ]:
missing_drop = missing.drop_nulls()
missing_fill = missing.with_columns(
    pl.col('int_data').fill_null(0),
    pl.col('float_data').fill_null(missing['float_data'].mean()),
    pl.col('category_data').fill_null('Unknown'),
    pl.col('string_data').fill_null('missing'),
)

print('original', missing.shape)
print('drop_nulls', missing_drop.shape)
print('fill_null', missing_fill.shape)
missing_fill


## 8. Clean Column Names

Normalize deliberately messy headers to lowercase snake_case.


In [ ]:
messy = metadata.rename(
    {
        'patient': 'Patient ID',
        'gender': 'Gender Code',
        'condition': 'Condition Group',
    }
)
rename_map = {
    name: name.strip().lower().replace(' ', '_')
    for name in messy.columns
}
clean = messy.rename(rename_map)
clean.head()


## 9. Filter with Combined Conditions

Reproduce a two-condition filter in one `.filter()` call.


In [ ]:
combined = patients.filter(
    (pl.col('temperature') > 38.5) & (pl.col('gender') == 'F')
)
combined.head()


## 10. Sort and Rank

Sort by temperature and add a dense rank column.


In [ ]:
ranked = patients.sort('temperature', descending=True).with_columns(
    pl.col('temperature').rank(method='dense', descending=True).alias('temp_rank')
)
ranked.select(['patient', 'temperature', 'temp_rank']).head(10)


## 11. Group and Aggregate

Group by condition and summarize temperature.


In [ ]:
patients.group_by('condition').agg(
    pl.len().alias('count'),
    pl.col('temperature').mean().alias('mean_temperature'),
    pl.col('temperature').std().alias('std_temperature'),
).sort('condition')


## 12. Build a Pivot Table

Compute mean temperature by condition and gender.


In [ ]:
patients.pivot(
    on='gender',
    index='condition',
    values='temperature',
    aggregate_function='mean',
).sort('condition')


## 13. Pivot with Duplicate Keys

Show why duplicate key pairs need an aggregation.


In [ ]:
duplicates = pl.DataFrame(
    {
        'patient': [1, 1, 1, 2],
        'measure': ['temp', 'temp', 'dose', 'temp'],
        'value': [38.1, 38.5, 2.0, 37.9],
    }
)

try:
    duplicates.pivot(on='measure', index='patient', values='value')
except Exception as exc:
    print(type(exc).__name__, exc)

duplicates.pivot(
    on='measure',
    index='patient',
    values='value',
    aggregate_function='mean',
)


## 14. Long to Wide

Reshape a tidy long table to wide format with `pivot()`.


In [ ]:
long_df = pl.DataFrame(
    {
        'patient': [1, 1, 2, 2],
        'day': ['day1', 'day2', 'day1', 'day2'],
        'temperature': [38.3, 38.5, 37.9, 38.0],
    }
)
long_df.pivot(on='day', index='patient', values='temperature').sort('patient')


## 15. Wide to Long

Convert repeated measurement columns to tidy long format.


In [ ]:
wide_df = pl.DataFrame(
    {
        'patient': [1, 2, 3],
        'day1': [38.3, 37.9, 38.1],
        'day2': [38.5, 38.0, 38.2],
        'day3': [38.1, 37.8, 38.0],
    }
)
wide_df.unpivot(index='patient', variable_name='day', value_name='temperature')


## 16. Classify with `when`/`then`

Map temperatures to categories with Polars expressions.


In [ ]:
classified = patients.with_columns(
    pl.when(pl.col('temperature') < 38.0)
    .then(pl.lit('normal'))
    .when(pl.col('temperature') < 38.5)
    .then(pl.lit('elevated'))
    .otherwise(pl.lit('high'))
    .alias('temp_band')
)
classified.select(['patient', 'temperature', 'temp_band']).head(10)


## 17. Reuse Expressions

Define the classification expression once and reuse it.


In [ ]:
temp_band_expr = (
    pl.when(pl.col('temperature') < 38.0)
    .then(pl.lit('normal'))
    .when(pl.col('temperature') < 38.5)
    .then(pl.lit('elevated'))
    .otherwise(pl.lit('high'))
    .alias('temp_band')
)

patients.with_columns(temp_band_expr).select(['patient', 'temperature', 'temp_band']).head(10)


## 18. Build a Method Chain

Join, filter, aggregate, and sort in one readable chain.


In [ ]:
summary = (
    experiment.join(metadata, on='patient', how='left')
    .filter(pl.col('temperature') > 38.0)
    .group_by('condition')
    .agg(pl.col('temperature').mean().alias('mean_temperature'))
    .sort('mean_temperature', descending=True)
)
summary


## 19. Read Generated CSV Data

Show the import side of the benchmark CSV generator with a tiny inline CSV.


In [ ]:
csv_text = """timestamp,C1,C2
2024-01-01 00:00:00,1.0,2.0
2024-01-01 00:00:01,1.1,2.1
"""
small_csv = pl.read_csv(StringIO(csv_text), try_parse_dates=True)
print(small_csv.schema)
small_csv


## 20. Mini Analysis

Join, summarize mean temperature by condition and gender, then plot.


In [ ]:
summary = (
    patients.group_by(['condition', 'gender'])
    .agg(pl.col('temperature').mean().alias('mean_temperature'))
    .sort(['condition', 'gender'])
    .pivot(on='gender', index='condition', values='mean_temperature')
    .sort('condition')
)

summary_pd = summary.to_pandas().set_index('condition')
ax = summary_pd.plot(kind='bar', title='Mean temperature by condition and gender', ylabel='temperature')
plt.tight_layout()
summary


## Notes for Instructors

- The overall flow mirrors the pandas solutions notebook so learners can compare syntax and outputs side by side.
- Exercises 6 and 7 use a synthetic null-containing table because the `polars/data/` directory only contains the two patient spreadsheets.
- Exercise 13 highlights a Polars-specific detail: there is one `pivot()` API, and duplicate keys require an explicit aggregation.
